# **Tarefa CIFAR-10**

O CIFAR-10 é um conjunto de dados amplamente utilizado para pesquisa em visão computacional e aprendizado de máquina. Foi criado por Alex Krizhevsky, Vinod Nair e Geoffrey Hinton e faz parte de um projeto maior do Instituto Canadense de Pesquisas Avançadas (CIFAR, na sigla em inglês), daí o nome.

## **Características principais**
* **Imagens:** 60.000 imagens coloridas (RGB) de 32×32 pixels.

* **Divisão:** 50.000 imagens para treino e 10.000 para teste.

* **Classes:** 10 categorias mutuamente exclusivas, cada uma com 6.000 imagens.

* **Rótulos**: As classes são: avião, automóvel, pássaro, gato, veado, cachorro, sapo, cavalo, navio e caminhão.




## **Desenvolva**


Construa uma rede neural convolucional com um número adequado de camadas convolucionais e de camadas ocultas na rede densa. Para o treinamento dessa rede, utilize 75% dos dados (treinamento) e, para a avaliação, os 25% restantes (teste). A seguir, desenvolva o que se pede:

**Item 1)(1,5)** Em uma mesma imagem, plote a curva da função de perda (*loss function*) e da acurácia (*accuracy*) em função do número de épocas.  
*Dica:* `historico = rede_neural_convolucional.fit(X_treinamento, y_treinamento, batch_size=128, epochs=?, validation_data=(X_teste, y_teste))`

**Item 2)(1,5)** Avalie o desempenho da rede neural na base de teste (`X_teste`, `y_teste`) por meio do comando `evaluate`.  
*Dica:* `resultado = rede_neural_convolucional.evaluate(X_teste, y_teste)`

**Item 3)(2,0)** Exiba, em uma única imagem, o resultado de 25 previsões, mostrando a imagem original, o rótulo correto e o rótulo previsto.  
*Dica:* Consulte o exercício resolvido em sala sobre o conjunto Fashion-MNIST.

**Item 4)(2,5)** Construa a matriz de confusão para o problema.  
*Dica:* Generalize a ideia apresentada no arquivo `Exemplo1_A2.1_MLP_classificadora_multiclasse_Iris_A.ipynb` da aula de classificação.

**Item 5)(2,5)** Realize um teste com validação cruzada k-fold e, em seguida, avalie o desempenho da rede neural nesse procedimento.

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.layers import InputLayer
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import KFold


In [ ]:

(X_treinamento, y_treinamento), (X_teste, y_teste) = datasets.cifar10.load_data()

In [ ]:
X_treinamento.shape

# **Pré-processamento**

In [ ]:
X_treinamento = X_treinamento.reshape(X_treinamento.shape[0],32,32,3)

X_teste = X_teste.reshape(X_teste.shape[0],32,32,3)

In [ ]:

X_treinamento = X_treinamento.astype('float32') / 255
X_teste = X_teste.astype('float32') / 255

In [ ]:

y_treinamento = tf.keras.utils.to_categorical(y_treinamento, 10)
y_teste = tf.keras.utils.to_categorical(y_teste, 10)

In [ ]:

class_names = ['Avião', 'Automóvel', 'Pássaro', 'Gato', 'Cervo',
               'Cachorro', 'Sapo', 'Cavalo', 'Navio', 'Caminhão']

## **Construindo a rede neural**

In [ ]:

rede_neural_convolucional = models.Sequential()

rede_neural_convolucional.add(InputLayer(shape=(32,32,3)))
rede_neural_convolucional.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
rede_neural_convolucional.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
rede_neural_convolucional.add(layers.MaxPooling2D((2, 2)))
rede_neural_convolucional.add(layers.Dropout(0.25))

rede_neural_convolucional.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
rede_neural_convolucional.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
rede_neural_convolucional.add(layers.MaxPooling2D((2, 2)))
rede_neural_convolucional.add(layers.Dropout(0.25))

rede_neural_convolucional.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
rede_neural_convolucional.add(layers.MaxPooling2D((2, 2)))
rede_neural_convolucional.add(layers.Dropout(0.3))

rede_neural_convolucional.add(layers.Flatten())
rede_neural_convolucional.add(layers.Dense(256, activation='relu'))
rede_neural_convolucional.add(layers.Dropout(0.4))
rede_neural_convolucional.add(layers.Dense(10, activation='softmax'))


In [ ]:
rede_neural_convolucional.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
rede_neural_convolucional.summary()

In [ ]:
# Treinar o modelo
historico_treinamento = rede_neural_convolucional.fit(X_treinamento, y_treinamento,
                    epochs=50,
                    batch_size=64,
                    validation_data=(X_teste, y_teste),
                    verbose=1)

## **Item 1 - Curvas de perda e acurácia**

In [ ]:
epocas = range(1, len(historico_treinamento.history['loss']) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epocas, historico_treinamento.history['loss'], label='Treino')
plt.plot(epocas, historico_treinamento.history['val_loss'], label='Validação')
plt.title('Loss por época')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epocas, historico_treinamento.history['accuracy'], label='Treino')
plt.plot(epocas, historico_treinamento.history['val_accuracy'], label='Validação')
plt.title('Acurácia por época')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()

plt.tight_layout()
plt.show()


## **Item 2 - Avaliação no conjunto de teste**

In [ ]:
resultado = rede_neural_convolucional.evaluate(X_teste, y_teste, verbose=0)
print(f'Loss no teste: {resultado[0]:.4f}')
print(f'Acurácia no teste: {resultado[1]:.4f}')


## **Item 3 - Exibir 25 previsões com rótulo correto e previsto**

In [ ]:
previsoes_prob = rede_neural_convolucional.predict(X_teste, verbose=0)
previsoes = np.argmax(previsoes_prob, axis=1)
rotulos_reais = np.argmax(y_teste, axis=1)

plt.figure(figsize=(15, 15))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_teste[i])
    correto = rotulos_reais[i]
    previsto = previsoes[i]
    cor_titulo = 'green' if correto == previsto else 'red'
    plt.title(f'R: {class_names[correto]}\nP: {class_names[previsto]}', color=cor_titulo, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()


## **Item 4 - Matriz de confusão**

In [ ]:
matriz_confusao = confusion_matrix(rotulos_reais, previsoes)

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=matriz_confusao, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('Matriz de Confusão - CIFAR-10')
plt.tight_layout()
plt.show()


## **Item 5 - Validação cruzada k-fold**

In [ ]:
def construir_modelo_cnn():
    modelo = models.Sequential([
        InputLayer(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax')
    ])

    modelo.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])
    return modelo

k = 5
epocas_kfold = 10
batch_size_kfold = 64

kfold = KFold(n_splits=k, shuffle=True, random_state=42)
acuracias_kfold = []
losses_kfold = []

for fold, (idx_treino, idx_validacao) in enumerate(kfold.split(X_treinamento), start=1):
    print(f'\nFold {fold}/{k}')

    X_tr, X_val = X_treinamento[idx_treino], X_treinamento[idx_validacao]
    y_tr, y_val = y_treinamento[idx_treino], y_treinamento[idx_validacao]

    modelo_fold = construir_modelo_cnn()

    modelo_fold.fit(X_tr, y_tr,
                    epochs=epocas_kfold,
                    batch_size=batch_size_kfold,
                    validation_data=(X_val, y_val),
                    verbose=0)

    loss_fold, acc_fold = modelo_fold.evaluate(X_val, y_val, verbose=0)
    losses_kfold.append(loss_fold)
    acuracias_kfold.append(acc_fold)

    print(f'Loss fold {fold}: {loss_fold:.4f}')
    print(f'Acurácia fold {fold}: {acc_fold:.4f}')

print('\nResumo k-fold:')
print(f'Acurácia média: {np.mean(acuracias_kfold):.4f} ± {np.std(acuracias_kfold):.4f}')
print(f'Loss média: {np.mean(losses_kfold):.4f} ± {np.std(losses_kfold):.4f}')
